# QLoRA training for binary solution fusion

This notebook trains Qwen3-4B-Instruct-2507 on merging-dataset JSON files and evaluates base versus adapted binary fusion trees. Labels are evaluator-accepted outputs; they do not by themselves prove valid reasoning or improvement over either candidate.

In [ ]:
# Point this at the uploaded/cloned project when it is not the current directory.
from pathlib import Path
import os
# The quantized model is pinned to one device; hide extra Kaggle GPUs before importing torch.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
REPO_DIR = Path('/kaggle/working/analogical_math_rag')
if (Path.cwd() / 'src').is_dir():
    REPO_DIR = Path.cwd()
if not (REPO_DIR / 'src').is_dir():
    raise FileNotFoundError(f'Project repository not found at {REPO_DIR}')
os.chdir(REPO_DIR)
# Kaggle provides CUDA PyTorch; keep it and install only the isolated workflow dependencies.
!pip install -q -r requirements-merging-finetuning.txt


In [ ]:
from pathlib import Path
import json, os
import numpy as np
import torch
from transformers import AutoTokenizer

from src.merging_finetuning import (
    QLoRAConfig, CompletionOnlyCollator, LocalFusionGenerator,
    build_evaluation_populations, compare_base_and_adapted_trees,
    evaluate_tree_trace, load_adapter_for_inference, load_qlora_model,
    prepare_merging_data, retrieve_exemplars_cpu, run_direct_fusion,
    smoke_test_training_step, summarize_evaluated_runs, tokenize_splits, train_qlora,
    upload_adapter_to_hub,
)
from src.utils import load_embedding_model, load_exemplar_corpus, load_json, save_json_atomic
from src.benchmark_data import load_target_benchmarks
from src.api_manager import AvalAIAPIManager
from config import CONFIG, setup_kaggle_mode

assert torch.cuda.is_available(), 'Select a Kaggle GPU accelerator before running this notebook.'
print(torch.cuda.get_device_name(0))


## Configuration
Edit these paths and run limits. Secrets are read from Kaggle Secrets or environment variables and are never printed.

In [ ]:
MERGING_JSON_PATHS = [
    '/kaggle/input/merging-dataset/merging_llama-3.2-11b_gpt-oss-20b_v1.json',
]
WORK_DIR = Path('/kaggle/working/merging-qwen3-4b-qlora')
MAX_LENGTH = 4096
SEED = 42
TRAIN = True
RESUME_CHECKPOINT = None  # e.g. '/kaggle/working/.../checkpoint-100'
EVAL_QUESTION_LIMIT = 5  # set None for the complete evaluation
GENERATION = {'temperature': 0.7, 'top_p': 0.8, 'max_new_tokens': 1024}
HF_UPLOAD_ENABLED = True
HF_TOKEN_SECRET_NAME = 'HF_TOKEN'  # Kaggle Secret or environment variable
HF_MODEL_REPO_ID = None  # None -> <authenticated-user>/merging-qwen3-4b-qlora
HF_MODEL_REPO_PRIVATE = True
HF_UPLOAD_COMMIT_MESSAGE = 'Upload trained merging QLoRA adapter'

def read_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception as exc:
        raise RuntimeError(f'Missing required secret {name}') from exc

qlora_config = QLoRAConfig(
    output_dir=str(WORK_DIR), max_length=MAX_LENGTH, epochs=3,
    learning_rate=1e-4, batch_size=1, gradient_accumulation_steps=8,
    lora_rank=16, lora_alpha=32, lora_dropout=0.05, seed=SEED,
)
WORK_DIR.mkdir(parents=True, exist_ok=True)


## Parse, deduplicate, split by question, and inspect token lengths

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(qlora_config.model_name, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
prepared = prepare_merging_data(
    MERGING_JSON_PATHS, WORK_DIR, tokenizer=tokenizer,
    max_length=MAX_LENGTH, seed=SEED,
)
print(json.dumps({
    'valid_records': sum(len(v) for v in prepared['splits'].values()),
    'split_sizes': {k: len(v) for k, v in prepared['splits'].items()},
    'parse_or_duplicate_failures': len(prepared['manifest']['parsing_failures']),
    'token_statistics': prepared['manifest']['token_statistics'],
}, indent=2))


## Load QLoRA model and run a real forward/backward smoke step

In [ ]:
model, tokenizer = load_qlora_model(qlora_config)
# Re-tokenize with the exact training tokenizer, then verify completion-only loss.
tokenized, token_report = tokenize_splits(prepared['splits'], tokenizer, MAX_LENGTH)
collator = CompletionOnlyCollator(tokenizer)
smoke_loss = smoke_test_training_step(model, collator, tokenized['train'][0])
print({'smoke_loss': smoke_loss, 'trainable_parameters': model.get_nb_trainable_parameters()})


## Train or resume, select the best adapter, and upload it
After successful training, the notebook uploads `best_adapter` when `HF_UPLOAD_ENABLED=True`. Add a write-enabled `HF_TOKEN` to Kaggle Secrets. The upload contains the QLoRA adapter and tokenizer files; the base model is referenced rather than duplicated.

In [ ]:
if TRAIN:
    trainer = train_qlora(
        model, tokenizer, tokenized, qlora_config,
        resume_from_checkpoint=RESUME_CHECKPOINT,
    )
    print('Best checkpoint:', trainer.state.best_model_checkpoint)
    if HF_UPLOAD_ENABLED:
        hf_token = read_secret(HF_TOKEN_SECRET_NAME)
        try:
            hub_result = upload_adapter_to_hub(
                WORK_DIR / 'best_adapter', hf_token, HF_MODEL_REPO_ID,
                private=HF_MODEL_REPO_PRIVATE,
                commit_message=HF_UPLOAD_COMMIT_MESSAGE,
            )
        finally:
            del hf_token
        print('Uploaded adapter:', hub_result['url'])


## Reload the saved adapter and configure local tree inference

In [ ]:
del model
torch.cuda.empty_cache()
model, tokenizer = load_adapter_for_inference(
    str(WORK_DIR / 'best_adapter'), qlora_config.model_name, gpu_index=0,
)
generator = LocalFusionGenerator(model, tokenizer, max_input_tokens=MAX_LENGTH)


## Load CPU retrieval resources and benchmark references
Ground truths stay in the evaluation records and are never passed to retrieval or generation.

In [ ]:
setup_kaggle_mode('/kaggle/working')
CONFIG.update({
    'TARGET_BENCHMARK': 'numina_hard', 'TARGET_BENCHMARKS': [],
    'BENCHMARK_MAX_QUESTIONS': None, 'EVAL_PARSE_BOXED_GROUND_TRUTH': True,
    'DEFAULT_EVALUATOR_TEMPERATURE': 0.0,
    'AVALAI_MODEL_NAME_EVALUATOR': 'openai/gpt-oss-20b',
})
exemplar_dataset = load_exemplar_corpus(CONFIG)
embedding_model = load_embedding_model(CONFIG)
embedded_exemplars = np.load(CONFIG['EMBEDDED_EXEMPLAR_CORPUS_QUESTIONS_PATH'])
exemplar_data = {
    'questions': list(exemplar_dataset['problem']),
    'solutions': list(exemplar_dataset['solution']),
}
benchmark_questions, benchmark_ground_truths, _ = load_target_benchmarks(
    CONFIG, exemplar_data, load_json_fn=load_json,
)
populations = build_evaluation_populations(
    prepared, benchmark_questions, benchmark_ground_truths,
)
print({name: len(rows) for name, rows in populations.items()})


## Initialize the existing AvalAI evaluator from a secret

In [ ]:
avalai_key = read_secret('AVALAI_API_KEY')
evaluator = AvalAIAPIManager(
    api_key_or_list=[avalai_key], base_url=CONFIG['AVALAI_BASE_URL'],
    model_quotas=CONFIG['AVALAI_MODEL_QUOTAS'], config=CONFIG,
)
del avalai_key


## Small three-mode tree demonstration

In [ ]:
demo_question = populations['remaining_benchmark'][0]['question']
demo_retrieved = retrieve_exemplars_cpu(
    demo_question, exemplar_data['questions'], exemplar_data['solutions'],
    embedded_exemplars, embedding_model, top_k=8,
)
demo = {
    'zero_shot': compare_base_and_adapted_trees(
        demo_question, generator, 'zero_shot', zero_shot_n=8,
        generation=GENERATION, seed=SEED,
    ),
    'retrieved': compare_base_and_adapted_trees(
        demo_question, generator, 'retrieved', retrieved_examples=demo_retrieved,
        generation=GENERATION, seed=SEED,
    ),
    'mixed': compare_base_and_adapted_trees(
        demo_question, generator, 'mixed', zero_shot_n=4,
        retrieved_examples=demo_retrieved[:4], generation=GENERATION, seed=SEED,
    ),
}
{mode: {arm: run['status'] for arm, run in arms.items()} for mode, arms in demo.items()}


## Evaluate both populations
The held-out accepted set also gets direct stored-pair fusion. Every tree arm shares identical cached leaves. Results are checkpointed after each question.

In [ ]:
def attach_evaluation(tree, ground_truth):
    return evaluate_tree_trace(tree, ground_truth, evaluator, CONFIG)

def evaluate_population(name, records, limit=None):
    runs = []
    selected = records if limit is None else records[:limit]
    for question_index, record in enumerate(selected):
        question, ground_truth = record['question'], record['ground_truth']
        retrieved = retrieve_exemplars_cpu(
            question, exemplar_data['questions'], exemplar_data['solutions'],
            embedded_exemplars, embedding_model, top_k=8,
        )
        for mode, kwargs in [
            ('zero_shot', {'zero_shot_n': 8}),
            ('retrieved', {'retrieved_examples': retrieved}),
            ('mixed', {'zero_shot_n': 4, 'retrieved_examples': retrieved[:4]}),
        ]:
            compared = compare_base_and_adapted_trees(
                question, generator, mode, generation=GENERATION,
                seed=SEED + question_index * 10000, **kwargs,
            )
            for arm, tree in compared.items():
                runs.append({
                    'population': name, 'benchmark_index': record['benchmark_index'],
                    'mode': mode, 'arm': arm, 'tree': tree,
                    'evaluation': attach_evaluation(tree, ground_truth),
                })
        if name == 'heldout_accepted':
            for arm, use_adapter in [('base', False), ('adapted', True)]:
                tree = run_direct_fusion(
                    question, record['candidate_solutions'], generator,
                    use_adapter=use_adapter, seed=SEED + question_index * 10000,
                    generation=GENERATION,
                )
                runs.append({
                    'population': name, 'benchmark_index': record['benchmark_index'],
                    'mode': 'direct', 'arm': arm, 'tree': tree,
                    'evaluation': attach_evaluation(tree, ground_truth),
                })
        if not save_json_atomic(runs, str(WORK_DIR / f'{name}_node_results.json')):
            raise OSError('Failed to checkpoint evaluation results')
    return runs

all_runs = []
for population_name, records in populations.items():
    all_runs.extend(evaluate_population(population_name, records, EVAL_QUESTION_LIMIT))


## Compact summaries by population, mode, and arm

In [ ]:
summary = {}
for population in sorted({run['population'] for run in all_runs}):
    for mode in sorted({run['mode'] for run in all_runs if run['population'] == population}):
        for arm in ('base', 'adapted'):
            subset = [run for run in all_runs if (run['population'], run['mode'], run['arm']) == (population, mode, arm)]
            if subset:
                summary[f'{population}/{mode}/{arm}'] = summarize_evaluated_runs(subset)
if not save_json_atomic(summary, str(WORK_DIR / 'evaluation_summary.json')):
    raise OSError('Failed to save evaluation summary')
print(json.dumps(summary, indent=2))
